In [1]:
#LANGUAGE
media_lang = 'tl'

#FILE LOCATION
file_path = r"G:\My Drive\SUBS\ALIGN\NON PRIO\BI_PRINSX_000032\Prinsesa Ng City Jail_ Xavier has a special visitor in the city jail! (Full Episode 32).mp4"

#H:\Other computers\TUF A16 Laptop\SUBS\

#MODEL
model_size = "large-v2"

In [2]:
# CODE BY github.com/nns2009
# FOR PROMPT: python refined_stable-ts.py <filepath> 100 43 0.1 0.5 0.3 2 22.01 1

import os 
os.environ['KMP_DUPLICATE_LIB_OK']='True'

import torch
import argparse
import stable_whisper
from stable_whisper.text_output import result_to_any, sec2srt

# From: https://stackoverflow.com/a/66909179
def int_or_skip(arg):
    try:
        return int(arg)  # try convert to int
    except ValueError:
        pass
    if arg == "_":
        return None
    raise argparse.ArgumentTypeError("must be an int or '_' for a default value")

# ----- Parse command line arguments -----
parser = argparse.ArgumentParser(description="Transcribed given video/audio file and create subtitles for it")
parser.add_argument("filename", type=str, help="Path to input video/audio file")
parser.add_argument("max_chars", type=int_or_skip, help="Maximum characters allowed in segment")
parser.add_argument("max_words", type=int_or_skip, help="Maximum words allowed in segment")
parser.add_argument("extend_start", type=float, help="Extend the start of all segments by this value (in seconds)")
parser.add_argument("extend_end", type=float, help="Extend the end of all segments by this value (in seconds)")
parser.add_argument("collapse_gaps_under", type=float, help="Collapse gaps between segmetns under a certain duration")
parser.add_argument("max_lines_per_segment", type=int, help="Max lines allowed per subtitle segment")
parser.add_argument("line_penalty", type=float, help="Penalty for each additional line (used to decide when to split segment into several lines)")
parser.add_argument("longest_line_char_penalty", type=float, help="Penalty for each character of the longest segment line (used to decide when to split segment into several lines)")

args = parser.parse_args([
    file_path, 
    '86', #MAX CHARS
    '30', #MAX WORDS
    '0', #EXTEND IN
    '0.5', #EXTEND OUT
    '0.3', #COLLAPSE GAPS
    '3', #MAX LINES PER SEGMENT
    '22.01', #LINE PENALTY
    '1' #LONGEST LINE CHARACTER PENALTY
])

print(args)
base_path = os.path.splitext(args.filename)[0]

Namespace(filename='G:\\My Drive\\SUBS\\ALIGN\\NON PRIO\\BI_PRINSX_000032\\Prinsesa Ng City Jail_ Xavier has a special visitor in the city jail! (Full Episode 32).mp4', max_chars=86, max_words=30, extend_start=0.0, extend_end=0.5, collapse_gaps_under=0.3, max_lines_per_segment=3, line_penalty=22.01, longest_line_char_penalty=1.0)


In [3]:
%%time

# ----- Load transcription data or transcribe -----
word_transcription_path = base_path + '.json'

if os.path.exists(word_transcription_path):
    print(f"Transcription data file found at {word_transcription_path}")
    result = stable_whisper.WhisperResult(word_transcription_path)
else:
    print(f"Can't find transcription data file at {word_transcription_path}. Starting transcribing ...")


    
    #FASTER-WHISPER
    model = stable_whisper.load_faster_whisper(model_size, device="cpu")
    
    result = model.transcribe_stable(args.filename, language=media_lang, vad=True, regroup=False)

    #Use IF low on VRAM
    #import gc; gc.collect(); torch.cuda.empty_cache(); del model
    
    #WHISPER AI
    #model = stable_whisper.load_model(model_size, device="cuda")
    #result = model.transcribe(args.filename, language=media_lang, vad=True, regroup=False)
    #model.refine(args.filename, result)
    # ^ Doesn't seem to do much, time refinements are minimal
    
    #DENOISER denoiser="demucs" - to separate vocals from music 
    
    result.save_as_json(word_transcription_path)
    
if args.max_chars or args.max_words:
    result.split_by_length(max_chars=args.max_chars, max_words=args.max_words)

# ----- Perform segment time extensions and anti-flickering (=closing the gaps) -----
extend_start = args.extend_start
extend_end = args.extend_end
for i in range(len(result) - 1):
    cur = result[i]
    next = result[i+1]
    
    # Testing
    #cur.start -= args.extend_start
    #cur.end += args.extend_end
    
    if next.start - cur.end < extend_start + extend_end: # Strict '<' to account for extend_start==extend_end==0
        # Not enough time to add the entire desired extensions -> add proportionally
        k = extend_end / (extend_start + extend_end) # cur extension factor
        mid = cur.end * (1-k) + next.start * k
        cur.end = next.start = mid
    else:
        # Add full desired extensions
        cur.end += extend_end
        next.start -= extend_start
        # Theoretically, because of floating-point errors, cur.end might be > next.start,
        # but it's going to be fixed in the next section anyway

        # <crossed> Strict '<' to save a bit of performance when both sides == 0 </crossed>
        # Non-strict because of theoretically possible floating-point errors:
        # 1.0 - 1.0000000000000001 computes to 0.0, which also should be fixed even with collapse_gaps_under==0
        if next.start - cur.end <= args.collapse_gaps_under:
            cur.end = next.start = (cur.end + next.start) / 2
if result: # (is not empty)
    result[0].start = max(0, result[0].start - extend_start)
    result[-1].end += extend_end
    # might go beyond the end of the file, but how would we know the file duration?

# ----- Export to SRT subtitles file -----
subtitles_path = base_path + '.srt'

max_lines_per_segment = args.max_lines_per_segment # Good value: 3 - don't bruteforce too much
line_penalty = args.line_penalty # Good value: 22.01 - .01 so "almost equal" prefer less lines
longest_line_char_penalty = args.longest_line_char_penalty # I'd just use 1.0 and choose line_penalty relative to it

# result.to_srt_vtt(subtitles_path, word_level = False)
# result.to_txt(subtitles_path, word_level = False)

def optimize_text(text: str):
    text = text.strip()
    words = text.split()

    # Compute prefix sums
    psum = [0]
    for w in words:
        psum += [psum[-1] + len(w) + 1] # +1 because of spaces

    bestScore = 10 ** 30
    bestSplit = None

    def backtrack(level, wordsUsed, maxLineLength, split):
        # level = Number of lines used so far
        nonlocal bestScore, bestSplit

        # Stop condition: All words used
        # Number of levels (=lines) might vary
        if wordsUsed == len(words):
            score = level * line_penalty + maxLineLength * longest_line_char_penalty
            if score < bestScore:
                bestScore = score
                bestSplit = split
            return
        
        if level + 1 == max_lines_per_segment:
            # Last level must include all remaining words
            backtrack(
                level + 1, len(words),
                max(maxLineLength, psum[len(words)] - psum[wordsUsed] - 1),
                split + [words[wordsUsed:]]
            )
            return
        
        # At least 1 word per line but up to all remaining words
        for levelWords in range(1, len(words)-wordsUsed + 1):
            backtrack(
                level + 1, wordsUsed + levelWords,
                max(maxLineLength, psum[wordsUsed + levelWords] - psum[wordsUsed] - 1),
                # -1 because N words only contain N-1 spaces between them
                split + [words[wordsUsed : wordsUsed+levelWords]]
            )

    backtrack(0, 0, 0, [])

    optimized = '\n'.join(' '.join(words) for words in bestSplit)
    if optimized != text:
        print('-----')
        print(text)
        print(optimized)
    return optimized

def segment2optimizedsrtblock(segment: dict, idx: int, strip=True) -> str:
    return f'{idx}\n{sec2srt(segment["start"])} --> {sec2srt(segment["end"])}\n' \
           f'{optimize_text(segment["text"])}'

def segments2blocks(segments):
    return '\n\n'.join(
        segment2optimizedsrtblock(s, i, strip=True)
        for i, s in enumerate(segments)
    )

result_to_any(
    result=result,
    filepath=subtitles_path,
    filetype='srt',
    segments2blocks=segments2blocks,
    word_level=False,
)


Can't find transcription data file at G:\My Drive\SUBS\ALIGN\NON PRIO\BI_PRINSX_000032\Prinsesa Ng City Jail_ Xavier has a special visitor in the city jail! (Full Episode 32).json. Starting transcribing ...
Detected Language: tagalog


Transcribe: 100%|██████████| 1486.87/1486.87 [49:29<00:00,  2.00s/sec]
VAD: 100%|██████████| 1486.87/1486.87 [00:16<00:00, 87.67sec/s]
Adjustment: 100%|██████████| 1482.22/1482.22 [00:00<00:00, 38742.43sec/s]


Saved: G:\My Drive\SUBS\ALIGN\NON PRIO\BI_PRINSX_000032\Prinsesa Ng City Jail_ Xavier has a special visitor in the city jail! (Full Episode 32).json
-----
Sir, nandito po si baby girl iniwan ni Divina.
Sir, nandito po si baby
girl iniwan ni Divina.
-----
Sinamantala mo pa ng lumaya habang nagkakagulo kami.
Sinamantala mo pa ng lumaya
habang nagkakagulo kami.
-----
Wala kang pakialam dahil hindi ka parte ng pamilyang ito.
Wala kang pakialam dahil hindi
ka parte ng pamilyang ito.
-----
Kaya naghahanap ako ng mas madandanda pa sana yung characters.
Kaya naghahanap ako ng mas
madandanda pa sana yung characters.
-----
I will make contract and let you know date for sign.
I will make contract and
let you know date for sign.
-----
Tingin ko po kasi kailangan na rin malaman ni Ati Lailani yung tungkol kay Xavier eh.
Tingin ko po kasi kailangan na rin malaman
ni Ati Lailani yung tungkol kay Xavier eh.
-----
Hindi naman po kaya ng konsensya ko na dead manhin na lang eh.
Hindi naman po kaya ng kon